## 訓練模型

In [4]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo26n.yaml")  # build a new model from YAML
model = YOLO("yolo26n.pt")  # load a pretrained model (recommended for training)
model = YOLO("yolo26n.yaml").load("yolo26n.pt")  # build from YAML and transfer weights

# Train the model
results = model.train(data="datasets\People Detection.v12i.yolo26\data.yaml", epochs=100, imgsz=640)

Transferred 708/708 items from pretrained weights
Ultralytics 8.4.53  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets\People Detection.v12i.yolo26\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-4, nbs=

## 圖片測試

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/detect/train-1/weights/best.pt")

results = model.predict(
    source="./test_set/2people.png",
    conf=0.25, # confidence threshold（信心閾值），超過保留
    save=True  # 是否儲存推論結果
)


image 1/1 c:\Users\User\Desktop\Github\deeplearn\test_set\2people.png: 544x640 2 faces, 2 persons, 46.3ms
Speed: 2.6ms preprocess, 46.3ms inference, 12.2ms postprocess per image at shape (1, 3, 544, 640)
Results saved to C:\Users\User\Desktop\Github\deeplearn\runs\detect\predict


## 影片測試

In [ ]:
from ultralytics import YOLO
import torch
import gc

model = YOLO("runs/detect/train-1/weights/best.pt")

for result in model.predict(
    source="./test_set/parkour.mp4",
    conf=0.25,       # confidence threshold（信心閾值），超過保留
    imgsz=640,       # inference image size（推論影像大小）
    save=True,       # 是否儲存推論結果
    stream=True,     # 串流模式(stream mode)，一次只處理一幀(frame)，不會把所有結果一次存進記憶體
    device=0,        # 如果要用第二張 GPU 就改 device=1
    half=True,       # NVIDIA GPU 可用，省顯存
    verbose=False    # 是否顯示詳細輸出資訊
):
    # 不要把 result 存進 list
    pass

torch.cuda.empty_cache() # 把 PyTorch「目前沒在用」的 GPU 記憶體釋放回 CUDA cache
gc.collect() # 強制執行垃圾回收

Results saved to C:\Users\User\Desktop\Github\deeplearn\runs\detect\predict-2


9

## 攝影機即時測試

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/detect/train-1/weights/best.pt")

model.predict(
    source=0,
    conf=0.25,
    show=True
)

## 正式驗證集評估

In [6]:
from ultralytics import YOLO

model = YOLO("runs/detect/train-1/weights/best.pt")

metrics = model.val(
    data="datasets/People Detection.v12i.yolo26/data.yaml",
    imgsz=640,
    conf=0.25,
    device=0
)

Ultralytics 8.4.53  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
YOLO26n summary (fused): 122 layers, 2,385,171 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.10.1 ms, read: 105.187.1 MB/s, size: 69.3 KB)
val: Scanning C:\Users\User\Desktop\Github\deeplearn\datasets\People Detection.v12i.yolo26\valid\labels.cache... 1431 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1431/1431  0.0s
val: C:\Users\User\Desktop\Github\deeplearn\datasets\People Detection.v12i.yolo26\valid\images\GX010023_frame_00025_right_jpg.rf.f39974421101490611a7618ef8d69ca4.jpg: 23 duplicate labels removed
WARNING Box and segment counts should be equal, but got len(segments) = 61, len(boxes) = 12598. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-9